# SISTEMAS DE BIG DATA - Examen 2ª Evaluación

### **Nombre**:

**INSTRUCCIONES**:

- Si realizas este examen desde tu ordenador debes **grabar la pantalla** con OBS Studio en formato MKV y entregar el vídeo junto con el examen. Para entregarlo, debes subirlo a OneDrive y adjuntar fichero de texto con la URL del recurso compartido.
- Si lo haces en un equipo del centro, grabaré yo la pantalla desde el ordenador del profesor.
- Debes contestar cada pregunta del examen en la celda del cuaderno Jupyter que hay después de cada pregunta. Si necesitas más celdas, puedes agregarlas a continuación de la que hay.
- Todo tu código **tiene que estár ejecutado**.
- La **entrega** del examen práctico se realizará en el canal de Teams habilitado a tal efecto y consistirá en:
  - Notebook de Jupyter (`.ipynb`).
  - Notebook exportado en formato Markdown (`.md`)
  - Fichero de texto (`.txt.`) con la URL al vídeo compartido en caso de haberlo hecho en tu ordenador.
  - Estos tres ficheros deberán entregarse en un único fichero comprimido en formato ZIP (`.zip`) con el nombre `{apellidos}, {nombre} - SBD Ev2`

Ejecuta la siguiente celda para generar el archivo `flota_rebelde.txt` con el que trabajás en este examen

In [1]:
%%writefile flota_rebelde.txt
name~!~base_asignada~!~naves_disponibles
CR90 corvette~!~Base Yavin 4~!~15
X-wing~!~Base Echo (Hoth)~!~120
Y-wing~!~Base Echo (Hoth)~!~45 naves
Millennium Falcon~!~Flota Nómada~!~1
A-wing~!~Base Endor~!~Error de sensor
Rebel transport~!~Punto de encuentro~!~Ocho
B-wing~!~Astillero Sullust~!~30
EF76 Nebulon-B escort frigate~!~Flota Nómada~!~4
Calamari Cruiser~!~Órbita Mon Cala~!~Desconocido
Star Destroyer~!~Hangar Secreto (Capturado)~!~2

Writing flota_rebelde.txt


## EXAMEN PRÁCTICO: Logística de la flota rebelde

Trabajas en el equipo de logística y suministros de la Alianza Rebelde. Recientemente, habéis recibido un archivo de texto con el inventario actual de naves espaciales disponibles en vuestras bases secretas. Sin embargo, este archivo fue generado por un sistema antiguo y contiene errores de formato.

Tu misión es extraer este inventario, limpiarlo y cruzarlo con el catálogo oficial de naves de la API de Star Wars (SWAPI) para calcular la capacidad de carga total de la flota.



### Apartado A: Ingesta y limpieza de la fuente estática (2.5 puntos)

El sistema legado ha exportado el inventario en el archivo `flota_rebelde.txt`. Al inspeccionarlo, notas que el separador de columnas es una secuencia extraña de caracteres (`~!~`) y que la columna numérica tiene datos corruptos.

1.  Carga el archivo `flota_rebelde.txt` en un DataFrame de Pandas. 
2.  La columna `naves_disponibles` contiene basura textual en algunas filas (ej. "5 naves", "Error"). Lee inicialmente esta columna como texto (cadena).
3.  Limpia la columna `naves_disponibles` forzando su conversión a tipo numérico. Asegúrate de transformar los textos irreconocibles en valores nulos (`NaN`).
4.  Elimina las filas que hayan quedado con valor nulo en dicha columna.

In [2]:
%cd examen2

/home/jovyan/work/examen2


In [3]:
import pandas as pd 

df = pd.read_csv("flota_rebelde.txt",sep=("~!~"))

s = df["naves_disponibles"]

df["naves_disponibles"] = pd.to_numeric(s, errors="coerce")

df = df.dropna()

df

/tmp/ipykernel_4234/3222233922.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv("flota_rebelde.txt",sep=("~!~"))


,name,base_asignada,naves_disponibles
0,CR90 corvette,Base Yavin 4,15.0
1,X-wing,Base Echo (Hoth),120.0
3,Millennium Falcon,Flota Nómada,1.0
6,B-wing,Astillero Sullust,30.0
7,EF76 Nebulon-B escort frigate,Flota Nómada,4.0
9,Star Destroyer,Hangar Secreto (Capturado),2.0



### Apartado B: Extracción de datos desde API REST (2.5 puntos)

Necesitamos obtener las especificaciones técnicas oficiales de todas las naves del universo Star Wars.

1.  Utilizando la librería `requests`, realiza peticiones `GET` al endpoint oficial de naves: `https://swapi.dev/api/starships/`.
2.  Carga la lista completa de naves en un único DataFrame de Pandas llamado `df_catalogo`.

*(Nota: Si no consigues hacer funcionar la petición a la API o la paginación, carga el archivo `swapi_starships_simulado.csv` en `df_catalogo` y continúa con el siguiente apartado).*

In [4]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [5]:
import requests
import pandas as pd

url = "https://swapi.dev/api/starships/"

lista_dfs = [] 

while url:
    response = requests.get(url)
    data = response.json()

    df_pagina = pd.json_normalize(data["results"])
    lista_dfs.append(df_pagina)
    

    url = data["next"]

df_catalogo = pd.concat(lista_dfs, ignore_index=True)
df_catalogo

,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,hyperdrive_rating,MGLT,starship_class,pilots,films,created,edited,url
0,CR90 corvette,CR90 corvette,Corellian Engineering Corporation,3500000,150,950,30-165,600,3000000,1 year,2.0,60,corvette,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T14:20:33.369000Z,2014-12-20T21:23:49.867000Z,https://swapi.dev/api/starships/2/
1,Star Destroyer,Imperial I-class Star Destroyer,Kuat Drive Yards,150000000,"1,600",975,"47,060",n/a,36000000,2 years,2.0,60,Star Destroyer,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T15:08:19.848000Z,2014-12-20T21:23:49.870000Z,https://swapi.dev/api/starships/3/
2,Sentinel-class landing craft,Sentinel-class landing craft,"Sienar Fleet Systems, Cyngus Spaceworks",240000,38,1000,5,75,180000,1 month,1.0,70,landing craft,[],[https://swapi.dev/api/films/1/],2014-12-10T15:48:00.586000Z,2014-12-20T21:23:49.873000Z,https://swapi.dev/api/starships/5/
3,Death Star,DS-1 Orbital Battle Station,"Imperial Department of Military Research, Sien...",1000000000000,120000,n/a,"342,953","843,342",1000000000000,3 years,4.0,10,Deep Space Mobile Battlestation,[],[https://swapi.dev/api/films/1/],2014-12-10T16:36:50.509000Z,2014-12-20T21:26:24.783000Z,https://swapi.dev/api/starships/9/
4,Millennium Falcon,YT-1300 light freighter,Corellian Engineering Corporation,100000,34.37,1050,4,6,100000,2 months,0.5,75,Light freighter,"[https://swapi.dev/api/people/13/, https://swa...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T16:59:45.094000Z,2014-12-20T21:23:49.880000Z,https://swapi.dev/api/starships/10/
5,Y-wing,BTL Y-wing,Koensayr Manufacturing,134999,14,1000km,2,0,110,1 week,1.0,80,assault starfighter,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-12T11:00:39.817000Z,2014-12-20T21:23:49.883000Z,https://swapi.dev/api/starships/11/
6,X-wing,T-65 X-wing,Incom Corporation,149999,12.5,1050,1,0,110,1 week,1.0,100,Starfighter,"[https://swapi.dev/api/people/1/, https://swap...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-12T11:19:05.340000Z,2014-12-20T21:23:49.886000Z,https://swapi.dev/api/starships/12/
7,TIE Advanced x1,Twin Ion Engine Advanced x1,Sienar Fleet Systems,unknown,9.2,1200,1,0,150,5 days,1.0,105,Starfighter,[https://swapi.dev/api/people/4/],[https://swapi.dev/api/films/1/],2014-12-12T11:21:32.991000Z,2014-12-20T21:23:49.889000Z,https://swapi.dev/api/starships/13/
8,Executor,Executor-class star dreadnought,"Kuat Drive Yards, Fondor Shipyards",1143350000,19000,n/a,"279,144",38000,250000000,6 years,2.0,40,Star dreadnought,[],"[https://swapi.dev/api/films/2/, https://swapi...",2014-12-15T12:31:42.547000Z,2014-12-20T21:23:49.893000Z,https://swapi.dev/api/starships/15/
9,Rebel transport,GR-75 medium transport,"Gallofree Yards, Inc.",unknown,90,650,6,90,19000000,6 months,4.0,20,Medium transport,[],"[https://swapi.dev/api/films/2/, https://swapi...",2014-12-15T12:34:52.264000Z,2014-12-20T21:23:49.895000Z,https://swapi.dev/api/starships/17/


### Apartado C: Transformación y cruce de datos (2.5 puntos)

Ahora debes unificar la información local con la oficial y hacer los cálculos logísticos.

1.  Realiza un cruce entre tu DataFrame del inventario limpio (Apartado A) y el DataFrame del catálogo oficial (Apartado B), utilizando el nombre de la nave como clave de unión.
2.  Crea una nueva columna calculada llamada `capacidad_total_flota`. Esta debe ser el resultado de multiplicar las `naves_disponibles` (de tu inventario) por la carga especificada en la API oficial.

In [6]:
df = df.set_index("name").join(df_catalogo.set_index("name"))

df["cargo_capacity"] = pd.to_numeric(df["cargo_capacity"])

df["capacidad_total_flota"] = df["naves_disponibles"] * df["cargo_capacity"]

df

,base_asignada,naves_disponibles,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,hyperdrive_rating,MGLT,starship_class,pilots,films,created,edited,url,capacidad_total_flota
name,,,,,,,,,,,,,,,,,,,,
CR90 corvette,Base Yavin 4,15.0,CR90 corvette,Corellian Engineering Corporation,3500000,150,950,30-165,600,3000000,1 year,2.0,60,corvette,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T14:20:33.369000Z,2014-12-20T21:23:49.867000Z,https://swapi.dev/api/starships/2/,45000000.0
X-wing,Base Echo (Hoth),120.0,T-65 X-wing,Incom Corporation,149999,12.5,1050,1,0,110,1 week,1.0,100,Starfighter,"[https://swapi.dev/api/people/1/, https://swap...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-12T11:19:05.340000Z,2014-12-20T21:23:49.886000Z,https://swapi.dev/api/starships/12/,13200.0
Millennium Falcon,Flota Nómada,1.0,YT-1300 light freighter,Corellian Engineering Corporation,100000,34.37,1050,4,6,100000,2 months,0.5,75,Light freighter,"[https://swapi.dev/api/people/13/, https://swa...","[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T16:59:45.094000Z,2014-12-20T21:23:49.880000Z,https://swapi.dev/api/starships/10/,100000.0
B-wing,Astillero Sullust,30.0,A/SF-01 B-wing starfighter,Slayn & Korpil,220000,16.9,950,1,0,45,1 week,2.0,91,Assault Starfighter,[],[https://swapi.dev/api/films/3/],2014-12-18T11:18:04.763000Z,2014-12-20T21:23:49.909000Z,https://swapi.dev/api/starships/29/,1350.0
EF76 Nebulon-B escort frigate,Flota Nómada,4.0,EF76 Nebulon-B escort frigate,Kuat Drive Yards,8500000,300,800,854,75,6000000,2 years,2.0,40,Escort ship,[],"[https://swapi.dev/api/films/2/, https://swapi...",2014-12-15T13:06:30.813000Z,2014-12-20T21:23:49.902000Z,https://swapi.dev/api/starships/23/,24000000.0
Star Destroyer,Hangar Secreto (Capturado),2.0,Imperial I-class Star Destroyer,Kuat Drive Yards,150000000,"1,600",975,"47,060",n/a,36000000,2 years,2.0,60,Star Destroyer,[],"[https://swapi.dev/api/films/1/, https://swapi...",2014-12-10T15:08:19.848000Z,2014-12-20T21:23:49.870000Z,https://swapi.dev/api/starships/3/,72000000.0


### Apartado D: Almacenamiento y salida (2.5 puntos)

El sistema de Inteligencia de Negocio y el Data Lake requieren que exportes los resultados en dos formatos distintos.

1.  **Capa de consumo (analistas):** exporta el DataFrame final resultante del Apartado C a un archivo CSV llamado `reporte_logistico.csv`. Debes asegurarte de utilizar coma (`,`) como separador y excluir explícitamente el índice numérico de Pandas.
2.  **Capa Plata (Data Lake):** para almacenar el histórico de forma eficiente para la CPU y en almacenamiento en frío, exporta el mismo DataFrame a formato **Apache Parquet**. Llama al archivo `historico_flota.parquet` y aplica compresión `gzip`.

In [7]:
df.to_csv("reporte_logistico.csv",sep=",")
df.to_parquet("historico_flota.parquet",compression="gzip")